### **Pipeline de datos textuales: calidad, particiones y carga eficiente en PyTorch**

Este cuaderno fue reestructurado para cubrir los siguientes temas: 

- Datos y corpus textuales,
- Partición **train/validation/test**,
- Métricas,
- Generalización,
- Overfitting,
- Sesgo-varianza,
- Calidad de datos,
- Limpieza,
- Deduplicación,
- Fairness,
- Contaminación (*data leakage*),
- y tratamiento básico de **PII**.

Además, mantiene una parte práctica de **PyTorch + `DataLoader` + `collate_fn`**.


Al terminar este cuaderno deberías poder:

1. Inspeccionar un corpus textual y detectar problemas de calidad
2. Anonimizar PII básica con reglas simples
3. Limpiar y normalizar texto
4. Deduplicar ejemplos para evitar contaminación entre particiones
5. Construir particiones `train`, `validation` y `test` de forma razonable
6. Implementar un `Dataset` y un `DataLoader` en PyTorch sin `torchtext`
7. Entrenar un clasificador textual sencillo
8. Medir desempeño con accuracy, precision, recall y macro-F1
9. Observar señales de generalización, overfitting y sesgo-varianza
10. Revisar métricas por grupo como chequeo básico de fairness.


In [ ]:
# Configuración general
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

import math
import random
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_colwidth", 120)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)


#### **1. Construcción de un corpus de ejemplo**

Para que el notebook sea **autocontenido** y no dependa de descargas externas, se construye un corpus sintético de mensajes de soporte.

La tarea será una clasificación binaria:

- `reclamo`
- `no_reclamo`

También añadimos:

- Una variable de grupo (`region`) para revisar fairness básico,
- Ejemplos con PII (email, teléfono, DNI),
- Duplicados exactos,
- Y casi-duplicados.

> En un trabajo real, esta celda se reemplaza por la carga de tu CSV, JSON o base de datos.


In [ ]:
regions = {
    "costa": {
        "reclamo": [
            "mi pedido llegó tarde y el soporte no responde",
            "la app se cae cada vez que intento pagar",
            "me cobraron dos veces por el mismo servicio",
            "el producto vino incompleto y nadie soluciona el caso",
            "el envío sigue retrasado desde ayer",
        ],
        "no_reclamo": [
            "quiero saber el estado de mi pedido",
            "gracias por la atención rápida en la app",
            "necesito información sobre los horarios de entrega",
            "me gustó la actualización del sistema",
            "cómo cambio la dirección de envío",
        ],
    },
    "sierra": {
        "reclamo": [
            "el sistema rechazó mi pago y no me dieron explicación",
            "no respetaron la garantía del equipo",
            "mi cuenta fue bloqueada sin aviso",
            "el paquete llegó dañado y con piezas faltantes",
            "la respuesta del call center fue demasiado lenta",
        ],
        "no_reclamo": [
            "quisiera consultar si hay stock disponible",
            "agradezco la orientación del asesor",
            "necesito ayuda para activar mi cuenta",
            "el proceso de registro fue claro",
            "podrían enviarme la boleta electrónica",
        ],
    },
    "selva": {
        "reclamo": [
            "la web no carga y perdí el turno de atención",
            "me cancelaron la compra sin motivo claro",
            "el reembolso todavía no aparece en mi banco",
            "nadie contesta mis mensajes desde hace días",
            "recibí otro producto distinto al solicitado",
        ],
        "no_reclamo": [
            "quiero consultar promociones vigentes",
            "muchas gracias por resolver mi duda",
            "cómo actualizo mis datos personales",
            "me interesa conocer los planes disponibles",
            "el servicio mejoró bastante esta semana",
        ],
    },
}

names = ["Ana", "Luis", "María", "Carlos", "Lucía", "Diego", "Rosa", "Jorge"]
products = ["plan premium", "router", "suscripción", "licencia", "pedido", "módulo"]
prefixes = ["hola equipo", "buenos días", "por favor", "estimados", "necesito apoyo", "consulta"]

rows = []
uid = 0

for region, templates in regions.items():
    for label in ["reclamo", "no_reclamo"]:
        for _ in range(12):
            base = random.choice(templates[label])
            product = random.choice(products)
            prefix = random.choice(prefixes)

            text = f"{prefix}, {base} con el {product}."

            if random.random() < 0.15:
                text += f" Mi nombre es {random.choice(names)}."
            if random.random() < 0.12:
                text += f" Escríbanme a usuario{uid}@correo.com."
            if random.random() < 0.10:
                text += f" Mi celular es 9{random.randint(10000000, 99999999)}."
            if random.random() < 0.06:
                text += f" Mi DNI es {random.randint(10000000, 99999999)}."

            rows.append(
                {
                    "id": uid,
                    "text": text,
                    "label": label,
                    "region": region,
                }
            )
            uid += 1

raw_df = pd.DataFrame(rows)

# Duplicados exactos
dup_rows = raw_df.sample(8, random_state=SEED).copy()
dup_rows["id"] = range(raw_df["id"].max() + 1, raw_df["id"].max() + 1 + len(dup_rows))
dup_rows["duplicate_source"] = "exact"

# Casi duplicados (mismo contenido con cambios de mayúsculas/puntuación)
near_rows = raw_df.sample(6, random_state=SEED + 1).copy()
near_rows["id"] = range(
    raw_df["id"].max() + 1 + len(dup_rows),
    raw_df["id"].max() + 1 + len(dup_rows) + len(near_rows),
)
near_rows["text"] = (
    near_rows["text"]
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.upper()
    + " !!!"
)
near_rows["duplicate_source"] = "near"

raw_df["duplicate_source"] = "original"
raw_df = pd.concat([raw_df, dup_rows, near_rows], ignore_index=True)

print("Tamaño del corpus crudo:", raw_df.shape)
raw_df.head(8)


#### **2. Inspección inicial de calidad del corpus**

Antes de tokenizar o entrenar, conviene mirar:

- Tamaño del corpus
- Balance de clases
- Balance por grupo
- Textos vacíos o nulos
- Longitud aproximada
- Posibles duplicados
- Y PII visible.


In [ ]:
summary = {
    "n_filas": len(raw_df),
    "nulos_en_text": int(raw_df["text"].isna().sum()),
    "textos_vacios": int((raw_df["text"].fillna("").str.strip() == "").sum()),
    "clases": raw_df["label"].value_counts().to_dict(),
    "regiones": raw_df["region"].value_counts().to_dict(),
}
summary


In [ ]:
raw_df["char_count"] = raw_df["text"].str.len()
raw_df["word_count_approx"] = raw_df["text"].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
raw_df["word_count_approx"].hist(ax=ax, bins=15)
ax.set_title("Distribución aproximada de longitud de texto")
ax.set_xlabel("Número de palabras")
ax.set_ylabel("Frecuencia")
plt.show()

raw_df[["id", "label", "region", "word_count_approx", "duplicate_source", "text"]].head(10)


#### **3. Detección y tratamiento básico de PII**

En esta semana no buscamos resolver anonimización perfecta, pero sí cubrir una capa mínima de limpieza:

- Correos electrónicos
- Teléfonos
- DNI de 8 dígitos.

La idea es **reemplazar**, no eliminar ciegamente, para conservar señal estructural.


In [ ]:
EMAIL_REGEX = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
PHONE_REGEX = re.compile(r"\b9\d{8}\b")
DNI_REGEX = re.compile(r"\b\d{8}\b")

def detect_pii(text: str) -> dict:
    text = text or ""
    return {
        "has_email": bool(EMAIL_REGEX.search(text)),
        "has_phone": bool(PHONE_REGEX.search(text)),
        "has_dni": bool(DNI_REGEX.search(text)),
    }

pii_flags = raw_df["text"].apply(detect_pii).apply(pd.Series)
raw_df = pd.concat([raw_df, pii_flags], axis=1)

pii_summary = {
    "emails": int(raw_df["has_email"].sum()),
    "phones": int(raw_df["has_phone"].sum()),
    "dnis": int(raw_df["has_dni"].sum()),
}
pii_summary


In [ ]:
raw_df.loc[
    raw_df[["has_email", "has_phone", "has_dni"]].any(axis=1),
    ["id", "label", "region", "text"]
].head(8)


#### **4. Limpieza y normalización**

Aplicaremos una limpieza simple :

1. Pasar a minúsculas
2. Anonimizar PII
3. Quitar tildes para simplificar el ejemplo
4. Normalizar espacios
5. Eliminar puntuación superficial
6. Conservar marcadores especiales como `<EMAIL>`, `<PHONE>`, `<DNI>`.

> En proyectos reales, algunas decisiones de limpieza pueden ser demasiado agresivas. Aquí las usamos con fines didácticos.


In [ ]:
def normalize_text(text: str) -> str:
    text = (text or "").lower().strip()

    # Reemplazo de PII
    text = EMAIL_REGEX.sub(" <EMAIL> ", text)
    text = PHONE_REGEX.sub(" <PHONE> ", text)
    text = DNI_REGEX.sub(" <DNI> ", text)

    # Normalización unicode
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    # Mantiene solo letras/números/marcadores especiales
    text = re.sub(r"[^a-zA-Z<>0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def simple_tokenize(text: str):
    return re.findall(r"<[^>]+>|[a-z0-9]+", text.lower())

raw_df["clean_text"] = raw_df["text"].apply(normalize_text)
raw_df["token_count"] = raw_df["clean_text"].str.split().str.len()

raw_df[["text", "clean_text"]].head(6)


#### **5. Deduplicación**

Un problema clásico en NLP es entrenar y evaluar sobre ejemplos repetidos o casi repetidos.  
Eso puede inflar métricas y ocultar problemas reales de generalización.

Como nuestros casi-duplicados difieren solo en mayúsculas y puntuación, tras la normalización terminan colapsando al mismo `clean_text`.


In [ ]:
duplicate_mask = raw_df.duplicated(subset=["clean_text"], keep=False)

dup_stats = {
    "filas_totales": len(raw_df),
    "textos_unicos_despues_de_normalizar": int(raw_df["clean_text"].nunique()),
    "filas_involucradas_en_duplicados": int(duplicate_mask.sum()),
}
dup_stats


In [ ]:
raw_df.loc[duplicate_mask, ["id", "duplicate_source", "label", "region", "clean_text"]].sort_values("clean_text").head(12)


#### **6. Contaminación entre particiones (*data leakage*)**

Primero haremos una partición **simple** sobre el corpus crudo.  
Luego veremos una partición más segura después de limpiar y deduplicar.

La pregunta clave es:

> ¿Aparece el mismo texto normalizado en más de un split?


In [ ]:
naive_train, naive_temp = train_test_split(
    raw_df,
    test_size=0.30,
    stratify=raw_df["label"],
    random_state=SEED,
)
naive_val, naive_test = train_test_split(
    naive_temp,
    test_size=0.50,
    stratify=naive_temp["label"],
    random_state=SEED,
)

def overlap_counts(parts: dict) -> dict:
    keys = list(parts.keys())
    sets = {k: set(v["clean_text"]) for k, v in parts.items()}
    out = {}
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = keys[i], keys[j]
            out[f"{a}_{b}"] = len(sets[a] & sets[b])
    return out

naive_overlap = overlap_counts(
    {"train": naive_train, "val": naive_val, "test": naive_test}
)
naive_overlap


#### **7. Partición segura: limpiar, deduplicar y estratificar**

Estrategia usada:

1. Trabajar con `clean_text`
2. Eliminar duplicados exactos sobre `clean_text`
3. Crear una clave de estratificación combinando `label` y `region`
4. Construir `train`, `validation` y `test`.

Esto cubre mejor:

- Calidad de corpus
- Deduplicación
- Contaminación
- Balance mínimo por clase y grupo.


In [ ]:
model_df = raw_df.drop_duplicates(subset=["clean_text"]).copy().reset_index(drop=True)
model_df["stratify_key"] = model_df["label"] + "_" + model_df["region"]

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["stratify_key"],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["stratify_key"],
    random_state=SEED,
)

safe_overlap = overlap_counts(
    {"train": train_df, "val": val_df, "test": test_df}
)

print("Tamaño train:", train_df.shape)
print("Tamaño val  :", val_df.shape)
print("Tamaño test :", test_df.shape)
print("Solapamientos entre splits:", safe_overlap)


In [ ]:
split_summary = pd.concat(
    {
        "train": train_df.groupby(["label", "region"]).size(),
        "val": val_df.groupby(["label", "region"]).size(),
        "test": test_df.groupby(["label", "region"]).size(),
    },
    axis=1,
).fillna(0).astype(int)

split_summary


#### **8. Construcción de vocabulario**

Usaremos un tokenizador muy simple basado en regex y construiremos el vocabulario manualmente a partir de `train`.

Esto es importante: **el vocabulario se ajusta solo con train**, no con validation ni test.


In [ ]:
SPECIAL_TOKENS = {"<pad>": 0, "<unk>": 1}

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(simple_tokenize(text))

    vocab = dict(SPECIAL_TOKENS)
    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab, counter

vocab, token_counter = build_vocab(train_df["clean_text"], min_freq=1)

print("Tamaño del vocabulario:", len(vocab))
print("Tokens más frecuentes:", token_counter.most_common(15))


#### **9. `Dataset`, `DataLoader` y `collate_fn` en PyTorch**

Aquí tratamos con los siguientes temas:

- `Dataset` personalizado,
- secuencias convertidas a índices,
- padding dinámico con `collate_fn`,
- longitudes por lote para usar un promedio enmascarado.


In [ ]:
label2id = {"no_reclamo": 0, "reclamo": 1}
id2label = {v: k for k, v in label2id.items()}

class TextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, vocab: dict):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        token_ids = [self.vocab.get(tok, self.vocab["<unk>"]) for tok in simple_tokenize(row["clean_text"])]

        return {
            "input_ids": torch.tensor(token_ids, dtype=torch.long),
            "label": torch.tensor(label2id[row["label"]], dtype=torch.long),
            "region": row["region"],
            "text": row["clean_text"],
        }

def collate_fn(batch):
    # Ordena por longitud es opcional, pero útil cuando luego se usan RNNs
    batch = sorted(batch, key=lambda x: len(x["input_ids"]), reverse=True)

    sequences = [item["input_ids"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)

    padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=SPECIAL_TOKENS["<pad>"],
    )

    return {
        "input_ids": padded,
        "lengths": lengths,
        "labels": labels,
        "regions": [item["region"] for item in batch],
        "texts": [item["text"] for item in batch],
    }

BATCH_SIZE = 16

train_loader = DataLoader(
    TextDataset(train_df, vocab),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    TextDataset(val_df, vocab),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    TextDataset(test_df, vocab),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)


In [ ]:
batch = next(iter(train_loader))

print("Forma input_ids:", batch["input_ids"].shape)
print("Forma labels   :", batch["labels"].shape)
print("Longitudes     :", batch["lengths"][:8].tolist())
print("Primer texto normalizado del lote:")
print(batch["texts"][0])


#### **10. Modelo base y métricas**

Para mantener el foco del cuaderno, usaremos un clasificador simple:

- Embedding
- Promedio de embeddings no-padding
- MLP pequeño de clasificación.

Esto basta para observar:

- Train vs validation
- Generalización
- Overfitting,
- Métricas en test.


In [ ]:
class AvgEmbClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, num_classes=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=SPECIAL_TOKENS["<pad>"])
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, lengths):
        emb = self.embedding(input_ids)                          # [B, T, D]
        mask = (input_ids != SPECIAL_TOKENS["<pad>"]).unsqueeze(-1)  # [B, T, 1]
        summed = (emb * mask).sum(dim=1)                        # [B, D]
        avg = summed / lengths.unsqueeze(-1).clamp(min=1)      # [B, D]
        return self.classifier(avg)

def compute_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

def evaluate_model(model, loader, return_predictions=False):
    criterion = nn.CrossEntropyLoss()
    model.eval()

    losses = []
    y_true, y_pred = [], []
    regions = []
    texts = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)

            losses.append(loss.item())
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
            regions.extend(batch["regions"])
            texts.extend(batch["texts"])

    metrics = {"loss": float(np.mean(losses))}
    metrics.update(compute_metrics(y_true, y_pred))

    if return_predictions:
        metrics["y_true"] = y_true
        metrics["y_pred"] = y_pred
        metrics["regions"] = regions
        metrics["texts"] = texts

    return metrics

def train_model(
    model,
    train_loader,
    val_loader,
    epochs=6,
    lr=1e-2,
    weight_decay=1e-4,
):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        train_losses = []
        train_true, train_pred = [], []

        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            preds = logits.argmax(dim=1)
            train_losses.append(loss.item())
            train_true.extend(labels.cpu().tolist())
            train_pred.extend(preds.cpu().tolist())

        train_metrics = {"train_loss": float(np.mean(train_losses))}
        train_metrics.update({
            f"train_{k}": v for k, v in compute_metrics(train_true, train_pred).items()
        })

        val_metrics = evaluate_model(model, val_loader)
        val_metrics = {f"val_{k}": v for k, v in val_metrics.items()}

        history.append({"epoch": epoch, **train_metrics, **val_metrics})

    return pd.DataFrame(history)

def plot_history(history_df, title="Curvas de entrenamiento"):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    ax.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    ax.plot(history_df["epoch"], history_df["train_f1_macro"], label="train_f1_macro")
    ax.plot(history_df["epoch"], history_df["val_f1_macro"], label="val_f1_macro")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    plt.show()


In [ ]:
main_model = AvgEmbClassifier(
    vocab_size=len(vocab),
    embed_dim=16,
    hidden_dim=32,
    dropout=0.2,
).to(DEVICE)

history_main = train_model(
    main_model,
    train_loader,
    val_loader,
    epochs=6,
    lr=1e-2,
    weight_decay=1e-4,
)

history_main


In [ ]:
plot_history(history_main, title="Modelo principal: train vs validation")


#### **11. Métricas en test e interpretación de errores**

Aquí no basta con una sola métrica.

Revisaremos:

- `accuracy`
- `precision_macro`
- `recall_macro`
- `f1_macro`
- matriz de confusión
- y algunos errores concretos.


In [ ]:
test_metrics = evaluate_model(main_model, test_loader, return_predictions=True)
{k: v for k, v in test_metrics.items() if k not in {"y_true", "y_pred", "regions", "texts"}}


In [ ]:
print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    target_names=[id2label[0], id2label[1]],
    zero_division=0,
))


In [ ]:
cm = confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"])

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm)
ax.set_title("Matriz de confusión (test)")
ax.set_xlabel("Predicción")
ax.set_ylabel("Etiqueta real")
ax.set_xticks([0, 1], [id2label[0], id2label[1]])
ax.set_yticks([0, 1], [id2label[0], id2label[1]])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.show()


In [ ]:
error_df = pd.DataFrame({
    "text": test_metrics["texts"],
    "region": test_metrics["regions"],
    "y_true": [id2label[i] for i in test_metrics["y_true"]],
    "y_pred": [id2label[i] for i in test_metrics["y_pred"]],
})

error_df[error_df["y_true"] != error_df["y_pred"]].head(10)


#### **12. Generalización**

Una señal útil:

- Si `train` mejora y `validation` también, el modelo está aprendiendo algo transferible,
- Si `train` sube mucho pero `validation` se estanca o empeora, aparece **overfitting**,
- Si ambos quedan bajos, puede haber **alto sesgo/underfitting**.


In [ ]:
history_main[[
    "epoch",
    "train_loss", "val_loss",
    "train_accuracy", "val_accuracy",
    "train_f1_macro", "val_f1_macro"
]]


#### **13. Fairness básico por grupo**

Aquí usamos `region` como grupo de análisis.  
No reemplaza un estudio serio de fairness, pero sí sirve como chequeo inicial para detectar si el sistema rinde mucho mejor en unos grupos que en otros.


In [ ]:
def group_metric_report(y_true, y_pred, groups):
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "group": groups})
    rows = []

    for group_name, subset in df.groupby("group"):
        metrics = compute_metrics(subset["y_true"], subset["y_pred"])
        rows.append({
            "group": group_name,
            "n": len(subset),
            **metrics,
        })

    return pd.DataFrame(rows).sort_values("group").reset_index(drop=True)

group_metric_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    test_metrics["regions"],
)


#### **14. Experimentos para ilustrar sesgo-varianza y overfitting**

Haremos dos experimentos:

##### **A. Modelo con mayor sesgo (underfitting)**
- Menos capacidad
- más regularización
- Pocas épocas.

##### **B. Modelo con mayor varianza/overfitting**
- Más capacidad
- Sin regularización
- Muy pocos datos de entrenamiento.


In [ ]:
small_train_df = train_df.sample(n=12, random_state=SEED)

small_train_loader = DataLoader(
    TextDataset(small_train_df, vocab),
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
)

underfit_model = AvgEmbClassifier(
    vocab_size=len(vocab),
    embed_dim=8,
    hidden_dim=8,
    dropout=0.5,
).to(DEVICE)

overfit_model = AvgEmbClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=128,
    dropout=0.0,
).to(DEVICE)

history_underfit = train_model(
    underfit_model,
    train_loader,
    val_loader,
    epochs=4,
    lr=1e-2,
    weight_decay=5e-3,
)

history_overfit = train_model(
    overfit_model,
    small_train_loader,
    val_loader,
    epochs=8,
    lr=2e-2,
    weight_decay=0.0,
)

print("Última fila underfit")
print(history_underfit.tail(1))

print("Última fila overfit")
print(history_overfit.tail(1))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_underfit["epoch"], history_underfit["train_f1_macro"], label="underfit_train_f1")
ax.plot(history_underfit["epoch"], history_underfit["val_f1_macro"], label="underfit_val_f1")
ax.plot(history_overfit["epoch"], history_overfit["train_f1_macro"], label="overfit_train_f1")
ax.plot(history_overfit["epoch"], history_overfit["val_f1_macro"], label="overfit_val_f1")
ax.set_title("Sesgo-varianza: comparación de curvas")
ax.set_xlabel("Epoch")
ax.legend()
plt.show()


#### **Lectura esperada de estos experimentos**

- **Alto sesgo/underfitting**: `train` y `validation` quedan relativamente bajos y cercanos.
- **Alta varianza/overfitting**: `train` sube mucho, pero `validation` no acompaña.
- **Generalización razonable**: la brecha `train-validation` existe, pero no explota y el rendimiento de validación sigue siendo útil.

En otras palabras:

- **sesgo** = el modelo no logra capturar suficiente señal;
- **varianza** = el modelo se adapta demasiado a peculiaridades del conjunto de entrenamiento.


#### **16. Cómo adaptar este cuaderno a un dataset real**

Cuando tengas un archivo real, cambia solo la parte de creación del corpus:

```python
real_df = pd.read_csv("tu_archivo.csv")
real_df = real_df.rename(columns={"texto": "text", "etiqueta": "label"})
real_df["region"] = real_df["region"].fillna("desconocida")
```

y luego reutiliza exactamente el resto del pipeline:

1. Detección de PII
2. Limpieza
3. Deduplicación
4. Partición segura
5. Vocabulario en `train`
6. `Dataset` + `DataLoader`
7. Entrenamiento y evaluación.

> La principal idea no es solo "cargar texto", sino **preparar el corpus de forma que la evaluación sea creíble**.


In [ ]:
# Plantilla mínima para adaptar el cuaderno a un dataset real
# Reemplaza las rutas y nombres de columnas según tu archivo.

# Ejemplo 1: CSV
# real_df = pd.read_csv("tu_dataset.csv")

# Ejemplo 2: Excel
# real_df = pd.read_excel("tu_dataset.xlsx")

# Ejemplo 3: JSON Lines
# real_df = pd.read_json("tu_dataset.jsonl", lines=True)

# Mapea tus columnas al esquema esperado por el cuaderno 
# Debes terminar con al menos:
# - text   : texto de entrada
# - label  : etiqueta objetivo
# - region : grupo opcional para fairness básico

# real_df = real_df.rename(columns={
#     "texto": "text",
#     "etiqueta": "label",
#     "zona": "region",
# })

# Si no tienes columna de grupo, crea una neutra
# if "region" not in real_df.columns:
#     real_df["region"] = "desconocida"

# Mantén solo las columnas necesarias
# raw_df = real_df[["text", "label", "region"]].copy()

# Crea un id si no existe
# raw_df["id"] = range(len(raw_df))
# raw_df["duplicate_source"] = ""

# Vista rápida
# print(raw_df.head())
# print(raw_df["label"].value_counts(dropna=False))
# print(raw_df["region"].value_counts(dropna=False))

# Luego ejecuta otra vez desde la sección:
# 2. Inspección inicial de calidad del corpus

#### **17. Ejercicios: adaptar el cuaderno a un dataset real**

Estos ejercicios están pensados para que reemplaces el corpus sintético por un dataset real sin romper el pipeline.

##### **Ejercicio 1. Mapeo del esquema**
Carga un archivo real y conviértelo al formato esperado por el cuaderno.

**Objetivo:** terminar con un `DataFrame` que tenga estas columnas:
- `text`
- `label`
- `region` (o una columna equivalente para análisis por grupo)

**Preguntas**
1. ¿Qué columna contiene el texto?
2. ¿Qué columna será la variable objetivo?
3. ¿Existe alguna columna útil para revisar fairness o segmentación?

##### **Ejercicio 2. Auditoría de calidad inicial**
Sobre tu dataset real, reporta:
- Número de filas
- Textos nulos o vacíos
- Distribución de clases
- Distribución por grupo
- Longitud media y mediana del texto.

**Entrega esperada:** una tabla de resumen y una breve interpretación.

##### **Ejercicio 3. Revisión de PII**
Usa las reglas del cuaderno para detectar:
- Correos
- Teléfonos
- Identificadores numéricos.

Luego responde:
1. ¿Las reglas actuales detectan bien la PII de tu dominio?
2. ¿Hay falsos positivos?
3. ¿Qué nuevos patrones deberías agregar?.

##### **Ejercicio 4. Limpieza y normalización**
Adapta `normalize_text` a tu problema.

**Decisiones a justificar**
- ¿Conviene quitar tildes?
- ¿Conviene eliminar puntuación?
- ¿Conviene conservar mayúsculas?
- ¿Debes preservar hashtags, URLs, emojis o menciones?.

##### **Ejercicio 5. Deduplicación**
Calcula:
- Número de duplicados exactos antes de limpiar
- Número de duplicados después de normalizar
- Ejemplos de casi-duplicados detectados.

**Pregunta clave:** ¿tu métrica de deduplicación debería basarse en texto exacto o en similitud?

##### **Ejercicio 6. Partición segura**
Construye `train`, `validation` y `test` sin contaminación.

**Tarea**
1. Decide una estrategia de estratificación.
2. Verifica si el mismo texto aparece en más de un split.
3. Explica por qué tu estrategia es válida para este caso.

##### **Ejercicio 7. Vocabulario y cobertura**
Después de construir el vocabulario solo con `train`, calcula:
- Tamaño del vocabulario,
- Tokens OOV en `validation` y `test`,
- Ejemplos de textos afectados por OOV.

**Discusión:** ¿tu tokenización simple sigue siendo suficiente?.

##### **Ejercicio 8. Línea base y evaluación**
Entrena el clasificador base y reporta:
- `accuracy`
- `precision_macro`
- `recall_macro`
- `f1_macro`
- Matriz de confusión.

Luego responde:
1. ¿La métrica principal debería ser accuracy o macro-F1?
2. ¿Qué errores son los más costosos en tu problema?.

##### **Ejercicio 9. Fairness básico**
Evalúa por grupo (`region` u otra variable equivalente).

**Analiza**
- Diferencias de F1 entre grupos
- Grupos con muy pocos ejemplos
- Posibles causas de brechas de rendimiento.

##### **Ejercicio 10. Informe técnico corto**
Redacta un informe breve con:
1. Descripción del dataset
2. Problemas de calidad detectados
3. Decisiones de limpieza
4. Estrategia de partición
5. Métricas finales
6. Limitaciones del experimento
7. Próximos pasos.

> Sugerencia: si tu dataset es muy pequeño, no interpretes métricas como si fueran conclusiones definitivas.

In [ ]:
# Ejercicio 1: carga y mapeo
# Completa esta celda con tu archivo real

# real_df = pd.read_csv("RUTA_DEL_ARCHIVO.csv")
# real_df = real_df.rename(columns={
#     "NOMBRE_COLUMNA_TEXTO": "text",
#     "NOMBRE_COLUMNA_LABEL": "label",
#     "NOMBRE_COLUMNA_GRUPO": "region",
# })

# if "region" not in real_df.columns:
#     real_df["region"] = "desconocida"

# raw_df = real_df[["text", "label", "region"]].copy()
# raw_df["id"] = range(len(raw_df))
# raw_df["duplicate_source"] = ""

# raw_df.head()


# Ejercicio 2: auditoría de calidad
# summary_real = {
#     "n_filas": len(raw_df),
#     "nulos_en_text": int(raw_df["text"].isna().sum()),
#     "textos_vacios": int((raw_df["text"].fillna("").str.strip() == "").sum()),
#     "clases": raw_df["label"].value_counts(dropna=False).to_dict(),
#     "grupos": raw_df["region"].value_counts(dropna=False).to_dict(),
#     "longitud_media": raw_df["text"].fillna("").str.split().str.len().mean(),
#     "longitud_mediana": raw_df["text"].fillna("").str.split().str.len().median(),
# }
# summary_real


# Ejercicio 3: detección de PII
# pii_flags_real = raw_df["text"].apply(detect_pii).apply(pd.Series)
# raw_df = pd.concat([raw_df, pii_flags_real], axis=1)
# raw_df.loc[
#     raw_df[["has_email", "has_phone", "has_dni"]].any(axis=1),
#     ["text", "has_email", "has_phone", "has_dni"]
# ].head(10)


# Ejercicio 4: limpieza
# raw_df["clean_text"] = raw_df["text"].apply(normalize_text)
# raw_df[["text", "clean_text"]].head(10)


# Ejercicio 5: deduplicación
# duplicate_mask_real = raw_df.duplicated(subset=["clean_text"], keep=False)
# dup_stats_real = {
#     "filas_totales": len(raw_df),
#     "textos_unicos_despues_de_normalizar": int(raw_df["clean_text"].nunique()),
#     "filas_involucradas_en_duplicados": int(duplicate_mask_real.sum()),
# }
# dup_stats_real


# Ejercicio 6: partición segura
# model_df = raw_df.drop_duplicates(subset=["clean_text"]).copy().reset_index(drop=True)
# model_df["stratify_key"] = model_df["label"].astype(str) + "_" + model_df["region"].astype(str)

# train_df, temp_df = train_test_split(
#     model_df,
#     test_size=0.30,
#     stratify=model_df["stratify_key"],
#     random_state=SEED,
# )
# val_df, test_df = train_test_split(
#     temp_df,
#     test_size=0.50,
#     stratify=temp_df["stratify_key"],
#     random_state=SEED,
# )

# print(train_df.shape, val_df.shape, test_df.shape)
# overlap_counts({"train": train_df, "val": val_df, "test": test_df})

# Ejercicio 7: vocabulario y cobertura
# vocab, token_counter = build_vocab(train_df["clean_text"], min_freq=1)
# print("Tamaño del vocabulario:", len(vocab))

# def oov_rate(texts, vocab):
#     total = 0
#     oov = 0
#     for text in texts:
#         toks = simple_tokenize(text)
#         total += len(toks)
#         oov += sum(tok not in vocab for tok in toks)
#     return {"total_tokens": total, "oov_tokens": oov, "oov_rate": (oov / total if total else 0.0)}

# print("OOV val :", oov_rate(val_df["clean_text"], vocab))
# print("OOV test:", oov_rate(test_df["clean_text"], vocab))

# Ejercicio 8: entrenamiento y evaluación
# train_loader = DataLoader(TextDataset(train_df, vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
# val_loader = DataLoader(TextDataset(val_df, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
# test_loader = DataLoader(TextDataset(test_df, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# real_model = AvgEmbClassifier(vocab_size=len(vocab), embed_dim=16, hidden_dim=32, dropout=0.2).to(DEVICE)
# history_real = train_model(real_model, train_loader, val_loader, epochs=6, lr=1e-2, weight_decay=1e-4)
# plot_history(history_real, title="Dataset real: train vs validation")

# real_test_metrics = evaluate_model(real_model, test_loader, return_predictions=True)
# {k: v for k, v in real_test_metrics.items() if k not in {"y_true", "y_pred", "regions", "texts"}}

# Ejercicio 9: fairness básico
# group_metric_report(
#     real_test_metrics["y_true"],
#     real_test_metrics["y_pred"],
#     real_test_metrics["regions"],
# )

In [ ]:
## Tus respuestas